In [1]:
import pandas as pd
import scanpy as sc
import pandas as pd
from recon.explore import Celltype
import numpy as np
import scanpy as sc  # single cell data
import pandas as pd  # data manipulation
import liana as li  # cell communication
import recon  # multilayer and perturbation prediction
import recon.data
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from utils import *

In [2]:
ccc_network = pd.read_csv("ccc_network.csv")

In [3]:
ccc_network["celltype_target"].unique()

array(['cDC', 'NK CD56bright', 'NK', 'pDC', 'NKT', 'CD8 Memory', 'MAIT',
       'Plasmablast', 'B Intermediate/Memory', 'CD16 Mono', 'HSPC',
       'CD14 Mono', 'CD4 Memory', 'ILC', 'Treg', 'CD8 Naive', 'B Naive',
       'CD4 Naive'], dtype=object)

In [4]:
file_name = "pbmc_scenicplus" # pbmc_dictys, pbmc_scenic, pbmc_scenicplus, pbmc_hummus
grn_path = "./GRMs_by_Pau_to_share/" + file_name + ".csv" 

grn = pd.read_csv(grn_path)
grn = grn.rename(columns={"score": "weight"})
print (grn)

      source                       cre   target    weight  pval
0       SPIB    chr2-17804303-17804803     SMC6  1.000000  0.01
1      CREB5    chr7-28409015-28409515    CREB5  0.999976  0.01
2       LEF1  chr4-108070747-108071247     LEF1  0.999951  0.01
3       SPIB  chr2-164840367-164840867   COBLL1  0.999927  0.01
4       SPIB    chr2-17804303-17804803     GEN1  0.999902  0.01
...      ...                       ...      ...       ...   ...
40935  TFDP2    chr4-83109795-83110295     HPSE -0.000098  0.01
40936   RARA   chr14-61343966-61344466    PRKCH -0.000073  0.01
40937   CHD2   chr21-44896348-44896848    ITGB2 -0.000049  0.01
40938   CHD2   chr10-71730327-71730827     PSAP -0.000024  0.01
40939   CHD2    chr3-98522082-98522582  ST3GAL6 -0.000000  0.01

[40940 rows x 5 columns]


In [5]:
grn = grn[grn["weight"] > 0].reset_index(drop = True)

In [6]:
receptor_genes = recon.data.load_receptor_genes("human_receptor_gene_from_NichenetPKN")
# for human, use "human_receptor_gene_from_NichenetPKN"

genes = np.unique(grn['source'].tolist() + grn['target'].tolist())
receptor_genes = receptor_genes[receptor_genes['target'].isin(genes)]
receptor_genes.head()

,source,target,weight
0,A1BG,A2M,0.022509
1,A1BG,ABCA1,0.005635
6,A1BG,ACOX1,0.005004
7,A1BG,ACSL1,0.006188
10,A1BG,ADK,0.005285


### Evaluation

In [7]:
# Mapping of ccc_network names to ground names is required.

dct_ct = {"CD4 Memory": 'CD4_Memory_T_cell', 
"NK" : "NK",
"pDC" : "pDC",
"B Intermediate/Memory" : "Intermediate_B_cell",
"CD8 Memory" : "CD8_Memory_T_cell",
"cDC" : "cDC",
"NKT" : "NKT", 
"NKS" : "NK",
'NK CD56bright' : "NK_CD56hi",
"MAIT" : "MAIT", 
"Plasmablast" : "Plasmablast",
"CD16 Mono" : "CD16_Mono",
"HSPC" : "HSPC",
"CD14 Mono" : "CD14_Mono",
'ILC' : "ILC", 
'Treg' : "Treg", 
'CD8 Naive' : "CD8_Naive_T_cell",
'B Naive' : "Naive_B_cell",
'CD4 Naive': "CD4_Naive_T_cell"
}

gt = pd.read_csv("./outputs/human_cytokine_dict_mini.csv")

#print (gt["celltype"].unique(), "\n", gt.columns, "\n ", gt["cytokine"].unique())

### Running RECON

In [8]:
table = pd.read_csv("important_cytokines_celltypes.csv")
table = table.rename(columns={'IL12A, IL12B': 'IL12A'})

ccc_network = ccc_network.map(lambda x: "NKS" if x == "NK" else x)
table = table.map(lambda x: "NKS" if x == "NK" else x)

In [22]:

full_dict = {}

for c in ['IL12A']: #table.columns: 

    
    seed = c # cytokine_to_gene[ct]
    print (seed)
    cell__types = table[c].dropna().tolist()
    
    # The seed has to change for any effects.
    direct_effect, indirect_effect = recon.explore.multicell_targets(
            seeds=[seed], 
            celltypes=cell__types,
            grn=grn,
            receptor_grn=receptor_genes,
            ccc=ccc_network,
            grn_graph_weighted=True,
            receptor_grn_graph_weighted=True,
            receptor_graph_weighted=False,
            cell_communication_graph_weighted=True,
            cell_communication_graph_directed=False,
            restart_proba=0.6,
            extend_seeds=True,
            njobs=15
        )
    direct_effect.to_csv("./Saved_results/effects_scenic_plus/" + seed + "_direct_effect.csv")
    indirect_effect.to_csv("./Saved_results/effects_scenic_plus/" + seed + "_indirect_effect.csv")


IL12A
Processing celltype 1/6: CD14 Mono
Processing celltype 2/6: CD16 Mono
Processing celltype 3/6: CD4 Naive
Processing celltype 4/6: NKS
Processing celltype 5/6: NK CD56bright
Processing celltype 6/6: cDC


/nfs/research/saezrodriguez/ajita/miniforge3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:132: UserWarning: 
                No receptor_graph provided,
                an empty receptor graph will be created.
                
/nfs/research/saezrodriguez/ajita/miniforge3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:474: UserWarning: The celltypes dictionary was converted to a list of Celltype objects.
The keys of the dictionary will be the celltype names.


Computing intracellular contributions and direct effect...
Computing intercellular contributions and indirect effect...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 7436.71it/s]
[Parallel(n_jobs=6)]: Done   2 out of   6 | elapsed:   16.7s remaining:   33.5s
[Parallel(n_jobs=6)]: Done   3 out of   6 | elapsed:   17.5s remaining:   17.5s
[Parallel(n_jobs=6)]: Done   4 out of   6 | elapsed:   18.3s remaining:    9.1s
[Parallel(n_jobs=6)]: Done   6 out of   6 | elapsed:   19.6s finished
